# Visualize Alignment Optimization Results

This notebook demonstrates how to load and visualize the optimization data captured during tilt-series alignment.

**Use this for:**
- Creating presentation-quality figures
- Exploring optimization behavior
- Analyzing precision evolution
- Comparing subvolumes across optimization steps

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch

from miss_alignment.alignment.visualize_alignment import load_optimization_data

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load Optimization Data

In [ ]:
# Set path to your optimization tracking directory
tracking_dir = Path("/path/to/output/optimization_steps")

# Load all step data
step_data = load_optimization_data(tracking_dir, load_subvolumes=True)

print(f"Loaded {len(step_data)} optimization steps")
print(f"Initial loss: {step_data[0].loss:.6f}")
print(f"Final loss: {step_data[-1].loss:.6f}")
print(f"Loss reduction: {step_data[0].loss - step_data[-1].loss:.6f}")

## 2. Plot Loss and Precision Evolution

In [ ]:
steps = [s.step for s in step_data]
losses = [s.loss for s in step_data]
mean_precisions = [s.mean_precision for s in step_data]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(steps, losses, 'o-', linewidth=2, markersize=8)
ax1.set_xlabel('Optimization Step', fontsize=14)
ax1.set_ylabel('Loss (Precision-Weighted Score)', fontsize=14)
ax1.set_title('Loss Evolution', fontsize=16, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Precision curve
ax2.plot(steps, mean_precisions, 's-', linewidth=2, markersize=8, color='C1')
ax2.set_xlabel('Optimization Step', fontsize=14)
ax2.set_ylabel('Mean Precision', fontsize=14)
ax2.set_title('Precision Evolution', fontsize=16, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Visualize Subvolumes at Different Steps

Compare reconstruction quality at different optimization steps.

In [ ]:
# Select steps to visualize (first, middle, last)
step_indices = [0, len(step_data) // 2, -1]
subvolume_idx = 0  # Which subvolume to display
z_slice = 48  # Which Z slice to show (middle of 96-pixel volume)

fig, axes = plt.subplots(1, len(step_indices), figsize=(15, 5))

for i, step_idx in enumerate(step_indices):
    data = step_data[step_idx]
    
    if data.subvolumes is not None and data.subvolumes.shape[0] > subvolume_idx:
        subvol = data.subvolumes[subvolume_idx, z_slice, :, :].numpy()
        precision = data.precisions[subvolume_idx].item() if data.precisions is not None else 0
        
        axes[i].imshow(subvol, cmap='gray')
        axes[i].set_title(
            f'Step {data.step}\nLoss: {data.loss:.4f}\nPrecision: {precision:.4f}',
            fontsize=12
        )
        axes[i].axis('off')
    else:
        axes[i].text(0.5, 0.5, 'No data', ha='center', va='center', transform=axes[i].transAxes)
        axes[i].axis('off')

plt.suptitle(f'Subvolume Evolution (Index {subvolume_idx}, Z-slice {z_slice})', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Analyze Precision Distribution

Visualize how model precision changes across the volume during optimization.

In [ ]:
# Get precision distributions at different steps
step_indices = [0, len(step_data) // 2, -1]

fig, axes = plt.subplots(1, len(step_indices), figsize=(15, 4))

for i, step_idx in enumerate(step_indices):
    data = step_data[step_idx]
    
    if data.precisions is not None:
        precisions = data.precisions.numpy()
        
        axes[i].hist(precisions, bins=30, alpha=0.7, edgecolor='black')
        axes[i].axvline(precisions.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {precisions.mean():.3f}')
        axes[i].set_xlabel('Precision', fontsize=12)
        axes[i].set_ylabel('Count', fontsize=12)
        axes[i].set_title(f'Step {data.step}', fontsize=12, fontweight='bold')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

plt.suptitle('Precision Distribution Evolution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Shift Evolution Analysis

Track how alignment shifts change during optimization.

In [ ]:
if step_data[0].shifts_x is not None:
    # Extract shifts
    shifts_x = torch.stack([s.shifts_x for s in step_data], dim=0)  # (n_steps, n_tilts)
    shifts_y = torch.stack([s.shifts_y for s in step_data], dim=0)
    
    # Compute shift magnitude
    shift_magnitude = torch.sqrt(shifts_x**2 + shifts_y**2)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot mean and individual tilts
    mean_magnitude = shift_magnitude.mean(dim=1)
    ax.plot(steps, mean_magnitude, linewidth=3, color='black', label='Mean', zorder=10)
    
    for i in range(shifts_x.shape[1]):
        ax.plot(steps, shift_magnitude[:, i], alpha=0.2, color='C0', linewidth=1)
    
    ax.set_xlabel('Optimization Step', fontsize=14)
    ax.set_ylabel('Shift Magnitude (Angstroms)', fontsize=14)
    ax.set_title('Shift Evolution During Optimization', fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No shift data available")

## 6. Create Publication-Quality Figure

Combine multiple visualizations into a single comprehensive figure.

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Loss curve (top left)
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(steps, losses, 'o-', linewidth=2, markersize=6)
ax1.set_xlabel('Step', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('A. Loss Evolution', fontsize=14, fontweight='bold', loc='left')
ax1.grid(True, alpha=0.3)

# Precision curve (top middle)
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(steps, mean_precisions, 's-', linewidth=2, markersize=6, color='C1')
ax2.set_xlabel('Step', fontsize=12)
ax2.set_ylabel('Mean Precision', fontsize=12)
ax2.set_title('B. Precision Evolution', fontsize=14, fontweight='bold', loc='left')
ax2.grid(True, alpha=0.3)

# Precision distribution (top right)
ax3 = fig.add_subplot(gs[0, 2])
if step_data[-1].precisions is not None:
    final_precisions = step_data[-1].precisions.numpy()
    ax3.hist(final_precisions, bins=30, alpha=0.7, edgecolor='black')
    ax3.axvline(final_precisions.mean(), color='red', linestyle='--', linewidth=2)
ax3.set_xlabel('Precision', fontsize=12)
ax3.set_ylabel('Count', fontsize=12)
ax3.set_title('C. Final Precision Distribution', fontsize=14, fontweight='bold', loc='left')
ax3.grid(True, alpha=0.3)

# Subvolume comparison (middle row)
subvolume_idx = 0
z_slice = 48
step_indices = [0, len(step_data) // 2, -1]
labels = ['D. Initial', 'E. Middle', 'F. Final']

for i, (step_idx, label) in enumerate(zip(step_indices, labels)):
    ax = fig.add_subplot(gs[1, i])
    data = step_data[step_idx]
    
    if data.subvolumes is not None and data.subvolumes.shape[0] > subvolume_idx:
        subvol = data.subvolumes[subvolume_idx, z_slice, :, :].numpy()
        im = ax.imshow(subvol, cmap='gray')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(f'{label}\nStep {data.step}, Loss: {data.loss:.4f}', fontsize=12, fontweight='bold', loc='left')
    ax.axis('off')

# Shift evolution (bottom row, spanning all columns)
ax7 = fig.add_subplot(gs[2, :])
if step_data[0].shifts_x is not None:
    shifts_x = torch.stack([s.shifts_x for s in step_data], dim=0)
    shifts_y = torch.stack([s.shifts_y for s in step_data], dim=0)
    shift_magnitude = torch.sqrt(shifts_x**2 + shifts_y**2)
    mean_magnitude = shift_magnitude.mean(dim=1)
    
    ax7.plot(steps, mean_magnitude, linewidth=3, color='black', label='Mean shift magnitude', zorder=10)
    for i in range(shifts_x.shape[1]):
        ax7.plot(steps, shift_magnitude[:, i], alpha=0.15, color='C0', linewidth=1)
    
    ax7.set_xlabel('Step', fontsize=12)
    ax7.set_ylabel('Shift Magnitude (Å)', fontsize=12)
    ax7.set_title('G. Shift Evolution', fontsize=14, fontweight='bold', loc='left')
    ax7.legend(fontsize=10)
    ax7.grid(True, alpha=0.3)

plt.suptitle('Alignment Optimization Progress', fontsize=18, fontweight='bold', y=0.995)
plt.savefig('optimization_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved as 'optimization_summary.png'")

## 7. Export Data for Further Analysis

In [ ]:
# Export summary statistics to CSV
import pandas as pd

summary_df = pd.DataFrame({
    'step': steps,
    'loss': losses,
    'mean_precision': mean_precisions,
})

summary_df.to_csv('optimization_summary.csv', index=False)
print("Summary statistics saved to 'optimization_summary.csv'")

# Display summary
summary_df

In [ ]:
# Alternative: Generate all 3D visualizations at once
from miss_alignment.alignment.visualize_alignment import create_3d_visualizations

# Uncomment to run:
# create_3d_visualizations(
#     tracking_dir=tracking_dir,
#     output_dir=Path("3d_visualizations"),
#     create_gifs=True,
#     gif_duration=500,
#     elev=30,
#     azim=45,
# )

print("To generate all 3D visualizations at once, uncomment and run the cell above.")

### 8.4 Alternative: Use the Convenience Function

Instead of manually creating plots, you can use the convenience function that generates everything at once.

In [ ]:
if has_position_data:
    from IPython.display import Image as IPImage, display
    
    print("Precision Evolution Animation:")
    display(IPImage(filename="precision_evolution.gif"))
    
    print("\nLoss Evolution Animation:")
    display(IPImage(filename="loss_evolution.gif"))

### 8.3 Display Generated GIFs

Display the generated animated GIFs in the notebook.

In [ ]:
if has_position_data:
    from miss_alignment.alignment.visualize_alignment import (
        generate_3d_animation_frames,
        save_animation_as_gif,
    )
    
    # Generate precision animation
    print("Generating precision animation frames...")
    precision_frames = generate_3d_animation_frames(
        steps_with_positions,
        value_key="precision",
        cmap="viridis",
        elev=30,
        azim=45,
    )
    
    print(f"Saving precision animation ({len(precision_frames)} frames)...")
    save_animation_as_gif(
        precision_frames,
        Path("precision_evolution.gif"),
        duration=500,  # milliseconds per frame
        dpi=100,
    )
    print("✓ Saved: precision_evolution.gif")
    
    # Generate loss animation
    print("\nGenerating loss animation frames...")
    loss_frames = generate_3d_animation_frames(
        steps_with_positions,
        value_key="loss",
        cmap="coolwarm",
        elev=30,
        azim=45,
    )
    
    print(f"Saving loss animation ({len(loss_frames)} frames)...")
    save_animation_as_gif(
        loss_frames,
        Path("loss_evolution.gif"),
        duration=500,
        dpi=100,
    )
    print("✓ Saved: loss_evolution.gif")

### 8.2 Generate Animated GIFs

Create animated GIFs showing how precision and loss evolve across optimization steps.

**Note:** This can take several minutes depending on the number of steps.

In [ ]:
if has_position_data:
    # Create loss plots
    if initial_step.scores is not None:
        fig = create_3d_scatter_plot(
            positions=initial_step.positions,
            values=initial_step.scores,
            title=f"Initial Loss Distribution (Step {initial_step.step})",
            colorbar_label="Loss",
            cmap="coolwarm",
            elev=30,
            azim=45,
        )
        plt.show()
        
        fig = create_3d_scatter_plot(
            positions=final_step.positions,
            values=final_step.scores,
            title=f"Final Loss Distribution (Step {final_step.step})",
            colorbar_label="Loss",
            cmap="coolwarm",
            elev=30,
            azim=45,
        )
        plt.show()

In [ ]:
if has_position_data:
    from miss_alignment.alignment.visualize_alignment import create_3d_scatter_plot
    
    # Select initial and final steps
    initial_step = steps_with_positions[0]
    final_step = steps_with_positions[-1]
    
    # Create precision plots
    if initial_step.precisions is not None:
        fig = create_3d_scatter_plot(
            positions=initial_step.positions,
            values=initial_step.precisions,
            title=f"Initial Precision Distribution (Step {initial_step.step})",
            colorbar_label="Precision",
            cmap="viridis",
            elev=30,
            azim=45,
        )
        plt.show()
        
        fig = create_3d_scatter_plot(
            positions=final_step.positions,
            values=final_step.precisions,
            title=f"Final Precision Distribution (Step {final_step.step})",
            colorbar_label="Precision",
            cmap="viridis",
            elev=30,
            azim=45,
        )
        plt.show()

### 8.1 Static 3D Scatter Plots

Create 3D scatter plots showing subvolume positions colored by precision or loss.

In [ ]:
# Check if we have position data
has_position_data = any(s.positions is not None for s in step_data)

if has_position_data:
    print("Position data available - can create 3D visualizations")
    
    # Filter steps with position data
    steps_with_positions = [s for s in step_data if s.positions is not None]
    print(f"Found {len(steps_with_positions)} steps with position data")
else:
    print("No position data found. Run alignment with tracking enabled and save_subvolumes=True")

## 8. 3D Visualization of Subvolume Positions

Visualize the spatial distribution of subvolumes in 3D and how precision/loss varies across the volume.